# LLM-powered Resume Screening and ATS Optimization Agent
Agente de Triagem de Currículos com Gemini ---
Desenvolvedora: Fabiana S.S

In [1]:
!pip install google-genai pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 2.3 MB/s eta 0:00:00


In [2]:
import os
import json
from google import genai
from google.genai import types
from google.colab import userdata
from google.colab import files
import pypdf

In [3]:
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

In [13]:
uploaded = files.upload()


# O nome do arquivo enviado precisa ser extraído do dicionário retornado pelo upload.
caminho_do_pdf = list(uploaded.keys())[0]
print(f"Arquivo recebido: {caminho_do_pdf}")

Saving curriculo ficitício -Gabriel de Alonco.pdf to curriculo ficitício -Gabriel de Alonco.pdf
Arquivo recebido: curriculo ficitício -Gabriel de Alonco.pdf


In [15]:
leitor_pdf = pypdf.PdfReader(caminho_do_pdf)
curriculo_texto = ""
for pagina in leitor_pdf.pages:
    texto_pagina = pagina.extract_text()
    if texto_pagina:
        curriculo_texto += texto_pagina + "\n"

if not curriculo_texto.strip():
    raise ValueError("Não foi possível extrair texto do PDF. Verifique se o arquivo não é uma imagem escaneada.")

print(curriculo_texto[:500], "...")

 
Gabriel  Vasconcelos  de  Alonço  Cientista  de  Dados  Principal  &  Especialista  em  IA  Sênior  São  Paulo,  SP  |  +55  (11)  99999-0000  |  gabriel.alonco@emailficticio.com  |  linkedin.com/in/gabriel-alonco-ficticio  
Resumo  Profissional  Cientista  de  Computação  com  mais  de  8  anos  de  experiência  no  desenvolvimento,  
treinamento
 
e
 
implantação
 
de
 
modelos
 
de
 
Inteligência
 
Artificial
 
e
 
Aprendizado
 
de
 
Máquina
 
(Machine
 
Learning).
 
Especialista
 
em
 
Pro ...


In [16]:
vaga_texto = """
Vaga: Analista de Dados Sênior
Requisitos: Domínio em ferramentas de Business Intelligence (BI), modelagem de dados e análise de indicadores.
Desejável: Experiência avançada com Power BI e integrações de IA.
"""

In [17]:
prompt_usuario = f"""
Aqui estão os insumos para análise:

[TEXTO DA VAGA]
{vaga_texto}

[CONTEÚDO DO CURRÍCULO EXTRAÍDO]
{curriculo_texto}
"""

In [18]:
system_instruction = """{
  "system_instruction": "Você é um Especialista em Recrutamento e Seleção (Tech Recruiter) sênior de nível internacional. Sua especialidade é analisar currículos, extrair dados com precisão cirúrgica e mapear competências reais contra descrições de vagas (Job Descriptions), fornecendo relatórios analíticos e adaptando a apresentação das informações sem nunca falsificar ou inventar dados.",
  "context": "O usuário fornecerá o texto de um currículo e o texto de uma vaga de emprego. O seu objetivo é processar esses dados através de um fluxo modular de trabalho para avaliar a aderência do candidato e gerar um currículo adaptado e otimizado para a vaga em questão.",
  "constraints": [
    "PROIBIDO INVENTAR OU ALUCINAR: Você não deve criar experiências, empresas, datas, projetos, tecnologias ou certificados que não estejam explicitamente mencionados no texto enviado.",
    "ADAPTAÇÃO BASEADA EM GROUNDING: Adaptar significa reordenar, mudar a ênfase de palavras-chave e reescrever bullet points para destacar impactos relevantes à vaga. Utilize os termos da vaga apenas quando houver equivalência real e comprovada no currículo fornecido.",
    "OMISSÃO ESTRATÉGICA: Se o currículo contiver experiências ou tecnologias totalmente irrelevantes para a vaga-alvo, reduza o espaço dedicado a elas, priorizando o que importa para o recrutador.",
    "IDIOMA: Se a vaga estiver em inglês e o currículo original em português, o novo currículo gerado e as análises devem ser traduzidos e entregues inteiramente em inglês."
  ],
  "processo_de_trabalho": [
    "Etapa 1 - Job Analysis: Identifique as hard skills obrigatórias, soft skills desejadas e os termos-chave essenciais.",
    "Etapa 2 - Resume Analysis: Varra o histórico do candidato buscando experiências, tecnologias ou sinônimos que atendam aos requisitos mapeados na Etapa 1.",
    "Etapa 3 - Match Engine: Realize uma análise quantitativa ponderada (Hard Skills 50%, Experiência 25%, Soft Skills 15%, Certificações 10%), calculando a porcentagem estimada de match, separando competências encontradas e destacando as lacunas.",
    "Etapa 4 - Verification & Formatting: Redija o novo documento focando nas conquistas e tecnologias correlatas à vaga e execute uma checagem rigorosa de integridade antes de formular o JSON final."
  ],
  "verification_loop": [
    "Confirme: Nenhuma empresa externa foi adicionada ao currículo?",
    "Confirme: Nenhuma tecnologia ou ferramenta foi inventada?",
    "Confirme: Nenhuma certificação ou curso inexistente foi criado?",
    "Confirme: Todas as informações presentes no texto final possuem origem estrita e exclusiva no currículo enviado?"
  ],
  "few_shot_examples": [
    {
      "cenario_1": {
        "entrada": {
          "curriculo": "Experiência sólida na construção de painéis gerenciais utilizando Power BI para o setor financeiro.",
          "vaga": "Requisitos: Domínio em ferramentas de Business Intelligence (BI) e análise de indicadores."
        },
        "saida_esperada": {
          "competencias": "* Business Intelligence (Power BI)"
        }
      }
    },
    {
      "cenario_2": {
        "entrada": {
          "curriculo": "Conhecimento conceitual em computação em nuvem obtido através de estudos autônomos. Sem experiência prática.",
          "vaga": "Desejável: Certificação AWS Certified Cloud Practitioner ou AWS Solutions Architect."
        },
        "saida_esperada_antialucinacao": {
          "competencias": "* Fundamentos de Cloud Computing",
          "nota_de_validacao": "A certificação AWS não foi adicionada ao perfil adaptado por não constar como emitida no currículo original."
        }
      }
    }
  ],
  "output_format": {
    "type": "JSON",
    "schema": {
      "relatorio_aderencia": {
        "compatibilidade_estimada_porcentagem": "number",
        "hard_skills_encontradas": [
          "string"
        ],
        "soft_skills_encontradas": [
          "string"
        ],
        "lacunas_identificadas": [
          "string"
        ]
      },
      "curriculo_adaptado": {
        "nome_candidato": "string",
        "dados_contato": "string",
        "resumo_profissional": "string",
        "competencias_tecnicas": [
          "string"
        ],
        "experiencia_profissional": [
          {
            "cargo": "string",
            "empresa": "string",
            "periodo": "string",
            "conquistas_e_atividades": [
              "string"
            ]
          }
        ],
        "educacao_e_certificacoes": [
          "string"
        ]
      }
    }
  }
}"""

In [20]:


generation_config = types.GenerateContentConfig(
    system_instruction=system_instruction,
    temperature=0.2,  # Mantém a IA rígida às regras anti-alucinação
    response_mime_type="application/json"
)


MODEL_NAME = "gemini-flash-latest"  # modelo definido explicitamente e usado de fato na chamada

In [21]:
print("Processando dados com o Gemini... Por favor, aguarde.")

try:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_usuario,
        config=generation_config
    )
except Exception as e:
    raise RuntimeError(f"Falha ao chamar a API do Gemini: {e}")

print("Resposta recebida.")

Processando dados com o Gemini... Por favor, aguarde.
Resposta recebida.


In [22]:
try:
    resultado = json.loads(response.text)
except json.JSONDecodeError:
    print("⚠️ A resposta não veio em JSON válido. Texto bruto retornado:")
    print(response.text)
    resultado = None

if resultado:
    print(json.dumps(resultado, ensure_ascii=False, indent=2))

{
  "relatorio_aderencia": {
    "compatibilidade_estimada_porcentagem": 65,
    "hard_skills_encontradas": [
      "Análise exploratória de dados",
      "SQL",
      "Integrações de IA (Especialista)",
      "Python e R",
      "Modelagem preditiva"
    ],
    "soft_skills_encontradas": [
      "Liderança técnica",
      "Mentoria de engenheiros",
      "Colaboração multidisciplinar",
      "Tradução de necessidades de negócios em requisitos técnicos"
    ],
    "lacunas_identificadas": [
      "Ausência de experiência explícita com a ferramenta Power BI (requisito desejável)",
      "Falta de menção direta a ferramentas tradicionais de Business Intelligence (BI) como Tableau ou Qlik",
      "Falta de foco explícito em modelagem de dados multidimensional (Star Schema, Snowflake) e análise de indicadores de negócios (KPIs) tradicionais"
    ]
  },
  "curriculo_adaptado": {
    "nome_candidato": "Gabriel Vasconcelos de Alonço",
    "dados_contato": "São Paulo, SP | +55 (11) 99999-0000 

In [23]:
if resultado:
    with open("resultado_analise.json", "w", encoding="utf-8") as f:
        json.dump(resultado, f, ensure_ascii=False, indent=2)
    files.download("resultado_analise.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>